#### ***Deep Learning Fundamentals Day 72 📊***
***
-  ***CNN for Image Classification***

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [3]:
# Datsets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image -> scale (0,1) -> normalize ->(-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR10 dataset
trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

In [4]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [5]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64)

- ***Building CNN***

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)   # flatten
        x = self.fc_layers(x)
        return x

In [7]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

- ***Training The CNN***

In [8]:
epochs = 10
for epoch in range(epochs):
     epoch_training_loss = 0.0

     for images,labels in trainloader:
          optimizer.zero_grad()

          output = model.forward(images) # FP
          loss = criterion(output,labels) # label fnx
          loss.backward() # BP
          optimizer.step() # output params
          rpoch_training_loss =+ loss.item()
     
     print(f"epoch:{epoch+1}/{epochs} & loss:{epoch_training_loss/len(trainloader)}")

epoch:1/10 & loss:0.0
epoch:2/10 & loss:0.0
epoch:3/10 & loss:0.0
epoch:4/10 & loss:0.0
epoch:5/10 & loss:0.0
epoch:6/10 & loss:0.0
epoch:7/10 & loss:0.0
epoch:8/10 & loss:0.0
epoch:9/10 & loss:0.0
epoch:10/10 & loss:0.0


- ***Evaluate Our CNN***

In [9]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images)   # better than model.forward()
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

accuracy = 100 * correct_labels / total_labels
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 75.76%
